# TitrateRL quickstart

In silico research only - not medical advice.

This notebook simulates the clinical titration algorithms on one virtual patient and plots the glucose trace and the dose trajectory.

In [ ]:
import matplotlib.pyplot as plt

from titraterl.cohort import default_cohort
from titraterl.evaluation import run_episode
from titraterl.policies import ATLantus, FixedDose, Insight, TreatToTarget

cohort = default_cohort()
patient = cohort.split("test")[3]
patient.patient_id, patient.params.body_weight_kg, patient.params.fpg_preinsulin

In [ ]:
results = {
    p.policy_id: run_episode(p, cohort, patient, seed=0)
    for p in (FixedDose(), ATLantus(), TreatToTarget(), Insight())
}
{k: round(100 * r.metrics["maintenance_tir"], 1) for k, r in results.items()}

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for name, res in results.items():
    fasting = [r.fasting_true for r in res.records]
    dose = [r.dose_u for r in res.records]
    ax1.plot(fasting, label=name)
    ax2.plot(dose, label=name)
ax1.axhspan(80, 130, color="green", alpha=0.1)
ax1.set_ylabel("Fasting glucose (mg/dL)")
ax2.set_ylabel("Basal dose (U)")
ax2.set_xlabel("Day")
ax1.legend()
fig.suptitle("Simulation result - not medical advice")
plt.show()

## The Gymnasium environment

`TitrateRL/T2D-v0` is registered on import. One step is one day; the action is a dose change of -4, -2, 0, +2 or +4 U.

In [ ]:
import gymnasium as gym

env = gym.make("TitrateRL/T2D-v0", reward="r3")
obs, info = env.reset(seed=0)
print(obs.shape, info["patient_id"], info["fasting_smbg"])